In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split
import quantstats as qs
from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import InverseVolatility, RiskBudgeting
from skfolio.preprocessing import prices_to_returns
import pandas as pd
import os
import glob
from skfolio import RiskMeasure, Population
from skfolio.prior import EmpiricalPrior
from skfolio.optimization import MeanRisk
from skfolio import RiskMeasure
import numpy as np
from sklearn.linear_model import LinearRegression
from skfolio.optimization import HierarchicalRiskParity
from statsmodels.tsa.stattools import coint
from itertools import combinations
from skfolio.moments import LedoitWolf

In [2]:
#Let's read a file to understand the file structure
df_AAPL=pd.read_parquet('/Users/astromainak1994gmail.com/Downloads/data_min/AAPL/AAPL_1min_10yr.parquet')
df_AAPL.head

<bound method NDFrame.head of         ticker  volume      open     close      high       low  \
0         AAPL     100  117.3700  117.3700  117.3700  117.3700   
1         AAPL     300  117.4900  117.4900  117.4900  117.4900   
2         AAPL     100  117.4900  117.4900  117.4900  117.4900   
3         AAPL     100  117.4900  117.4900  117.4900  117.4900   
4         AAPL     100  117.4600  117.4600  117.4600  117.4600   
...        ...     ...       ...       ...       ...       ...   
1807921   AAPL     125  278.4264  278.4264  278.4264  278.4264   
1807922   AAPL     105  278.4211  278.4211  278.4211  278.4211   
1807923   AAPL     695  278.3400  278.3900  278.3900  278.3393   
1807924   AAPL    1051  278.2200  278.3000  278.3400  278.2000   
1807925   AAPL     146  278.3400  278.3400  278.3400  278.3400   

                window_start  transactions  
0        1449046800000000000             1  
1        1449048720000000000             2  
2        1449049620000000000             1

In [3]:
# Load all 52 equity minute bar parquet files from Polygon.
# We use glob to recursively find all files matching the pattern.
# Each file lives in its own subfolder named after the ticker (e.g. data_min/AAPL/AAPL_1min_10yr.parquet).
# We extract the ticker name from the subfolder, load only the close price,
# and rename the column to the ticker so we can stack them side by side later.

file_pattern='/Users/astromainak1994gmail.com/Downloads/data_min/**/*_1min_10yr.parquet'
files=glob.glob(file_pattern)
index=[os.path.basename(os.path.dirname(f)) for f in files]
df_list=[pd.read_parquet(f).set_index('window_start')[['close']].rename(columns={'close':t}) for f, t in zip(files,index)]

In [4]:
# Stack all 52 close price series side by side into one price matrix.
# Rows = timestamps, columns = tickers.
# Polygon stores timestamps in UTC nanoseconds — we convert to datetime,
# localize to UTC, then convert to New York time to handle daylight saving correctly.
# We then filter to regular market hours only (9:30am - 4:00pm ET),
# dropping pre-market and after-hours noise.

final_df=pd.concat(df_list,axis=1)
final_df.index=pd.to_datetime(final_df.index, unit='ns').tz_localize('UTC').tz_convert('America/New_York')
final_df=final_df.between_time('9:30','16:00')

In [5]:
# Resample from 1-minute to 30-minute bars by taking the last price in each window.
# Minute resolution has significant microstructure noise (bid-ask bounce, sparse trades).
# 30-min bars preserve intraday structure while reducing noise.
# Forward-fill any remaining NaNs — sparse minute bars become clean 30-min bars.
# Compute two representations:
# - returns: percentage change bar-to-bar (used for correlation-based pairs)
# - log_prices: log of price levels (used for cointegration-based pairs)

final_df=final_df.resample('30 min').last()
final_df=final_df.ffill()
returns=final_df.pct_change()
log_prices=np.log(final_df)

In [6]:
# Sanity check — confirm zero NaNs in the price matrix before proceeding.
# Any NaN here would contaminate downstream signal and optimization.
print(log_prices.isna().sum().sum())

0


In [7]:
daily_prices = final_df.resample('1D').last().dropna(how='all')
log_prices_daily=np.log(daily_prices)

split_daily=int(len(log_prices_daily)*0.67)
print(split_daily)

log_prices_daily_train=log_prices_daily.iloc[:split_daily]
log_prices_daily_test=log_prices_daily.iloc[split_daily:]



2445


In [8]:
# Cointegration-based pair selection using the Engle-Granger test.
# Two price series are cointegrated if their spread is stationary —
# meaning they share a long-run equilibrium and deviations mean-revert.
# We test all 1,326 possible pairs (52 choose 2) on TRAINING data only.
# Using daily prices for cointegration — it is a long-run concept,
# and daily observations are more statistically independent than 30-min bars.
# Lower p-value = stronger cointegration evidence.
# Results sorted ascending so top pairs are most cointegrated.

results = {}
for i, j in combinations(log_prices_daily_train.columns, 2):
    stat, p_value, crit = coint(log_prices_daily_train[i], log_prices_daily_train[j])
    results[(i, j)] = p_value
coint_series = pd.Series(results).sort_values()

In [9]:
## Estimate the hedge ratio (beta) and intercept (alpha) for each of the top 20 cointegrated pairs.
# We regress log price of stock i on log price of stock j via OLS:
#     log(P_i) = alpha + beta * log(P_j) + residual
# Beta tells us how many units of j to hold per unit of i to construct the spread.
# The spread (regression residual) is the stationary, tradeable quantity.
# CRITICAL: coefficients are estimated on daily TRAINING data only and frozen for the test period.
# Re-estimating on test data would introduce look-ahead bias.

alphas={}
betas={}
for i, j in coint_series.head(20).index:
    log_prices_daily_train_i=log_prices_daily_train[i]
    log_prices_daily_train_j=log_prices_daily_train[j]
    model=LinearRegression()
    model.fit(log_prices_daily_train_j.values.reshape(-1,1),log_prices_daily_train_i)
    betas[(i,j)]=model.coef_[0]
    alphas[(i,j)]=model.intercept_


In [10]:
# COINTEGRATION STRATEGY: signal generation on 30-min bars with frozen daily parameters.
#
# For each pair, construct the spread using alpha/beta frozen from daily training data:
#     spread = log(P_i) - (alpha + beta * log(P_j))
#
# Signal (mean reversion):
#   - Z-score the spread with a 78-bar rolling window (~6 trading days)
#   - Z > +2 -> short the spread, Z < -2 -> long the spread
#   - Z clipped at ±3 to limit outliers
#
# P&L: pair_return[t] = signal[t-1] * (spread[t] - spread[t-1])
# shift(1) prevents look-ahead bias — signal observed at t-1, traded at t.
#
# Half-life diagnostic: regress spread changes on lagged spread level.
# The slope gives mean-reversion speed; half-life = -ln(2)/speed (÷13 for days).
# All pairs show 100+ day half-lives — too slow for a 30-min signal.
# This motivates the correlation-based approach next.

pair_returns_dict_train = {}
pair_returns_dict_test = {}
split_30_min=int(len(log_prices)*0.67)
log_prices_30min_train=log_prices.iloc[:split_30_min].dropna()
log_prices_30min_test=log_prices.iloc[split_30_min:].dropna()
for (i, j), beta, in betas.items():
    alpha=alphas[(i,j)]

    # Train period
    spread_train=log_prices_30min_train[i]-(alpha+beta*log_prices_30min_train[j])
    z_score_train_raw=(spread_train-spread_train.rolling(window=78).mean())/spread_train.rolling(window=78).std()
    z_score_train=np.clip(z_score_train_raw,-3,3)
    signal_train=pd.Series(np.where(z_score_train>2,-1,np.where(z_score_train<-2,1,0)),index=log_prices_30min_train.index)
    pair_returns_train=signal_train.shift(1)*spread_train.diff()
    pair_returns_dict_train[f'{i}_{j}']=pair_returns_train

    # Half-life diagnostic
    y = spread_train.diff().dropna()
    x = spread_train.shift(1).loc[y.index]
    speed = LinearRegression().fit(x.values.reshape(-1,1), y.values).coef_[0]
    print(f"{i}_{j} half-life: {-np.log(2)/speed/13:.0f} days")

    # Test period (frozen alpha/beta, fresh rolling z-score — no leakage)
    spread_test=log_prices_30min_test[i]-(alpha+beta*log_prices_30min_test[j])
    z_score_test=(spread_test-spread_test.rolling(window=78).mean())/spread_test.rolling(window=78).std()
    z_score_test=np.clip(z_score_test,-3,3)
    signal_test=pd.Series(np.where(z_score_test>2,-1,np.where(z_score_test<-2,1,0)),index=log_prices_30min_test.index)
    pair_returns_test=signal_test.shift(1)*spread_test.diff()
    pair_returns_dict_test[f'{i}_{j}']=pair_returns_test

pair_returns_train_df=pd.DataFrame(pair_returns_dict_train).dropna()
pair_returns_test_df=pd.DataFrame(pair_returns_dict_test).dropna()

WMT_NKE half-life: 147 days
MSFT_NKE half-life: 141 days
NKE_ADBE half-life: 153 days
HD_NKE half-life: 139 days
CRM_NKE half-life: 138 days
LRCX_NKE half-life: 202 days
TGT_NKE half-life: 142 days
MCD_NKE half-life: 219 days
FB_NKE half-life: 195 days
COST_NKE half-life: 177 days
ORCL_NKE half-life: 190 days
AVGO_NKE half-life: 219 days
JNJ_AVGO half-life: 108 days
AMAT_NKE half-life: 298 days
IBM_NKE half-life: 178 days
CRM_ADBE half-life: 142 days
CAT_NKE half-life: 312 days
AMZN_GOOGL half-life: 155 days
WMT_CRM half-life: 182 days
SBUX_NKE half-life: 201 days


In [11]:
# Portfolio construction with skfolio's RiskBudgeting optimizer.
# Each pair is treated as an asset; the optimizer allocates capital so that
# every pair contributes equally to total portfolio variance (risk parity).
# This avoids concentration in any single pair.
#
#
#   - fit() on TRAIN pair returns -> learns optimal weights
#   - predict() on TEST pair returns -> applies frozen weights out-of-sample
#
# Finally, generate a QuantStats tearsheet (Sharpe, drawdown, win rate, etc.)
# from the out-of-sample portfolio returns.
# Result: OOS annualized Sharpe ~0.2

model = RiskBudgeting(risk_measure=RiskMeasure.VARIANCE)
model.fit(pair_returns_train_df)
ptf = model.predict(pair_returns_test_df)
returns_series = pd.Series(ptf.returns, index=pair_returns_test_df.index)
qs.reports.html(returns_series, annualized_factor=252*13, output='pairs_trading_report_final.html')

In [12]:
# CORRELATION STRATEGY: pair selection on short-term return co-movement.
#
# Key insight from the cointegration attempt: cointegration captures long-run
# price relationships (half-lives of months), mismatched with a 30-min signal.
# For intraday trading, short-term RETURN correlation is the right selection
# criterion — correlated returns mean the stocks move together bar-to-bar.
#
# Compute the correlation matrix on 30-min TRAINING returns only (no leakage).
# Extract the upper triangle (k=1 excludes self-correlation and duplicates),
# rank all pairs, and keep the top 20 most correlated.

returns_30min_train=returns.iloc[:split_30_min].dropna()
returns_30min_test=returns.iloc[split_30_min:].dropna()
corr_matrix=returns_30min_train.corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_pairs = upper_tri.stack().sort_values(ascending=False)
N = 20
top_N = corr_pairs.head(N)
print(top_N)

BAC   JPM     0.885566
      C       0.856993
C     JPM     0.851520
AMAT  LRCX    0.843398
GS    JPM     0.808982
CVX   XOM     0.800052
GS    BAC     0.798521
WFC   BAC     0.796874
GS    C       0.793558
WFC   JPM     0.790610
      C       0.777954
CVX   COP     0.753637
XOM   COP     0.752128
KSS   M       0.740952
SLB   COP     0.735615
XOM   OXY     0.728943
SLB   OXY     0.726139
OXY   COP     0.725172
GS    WFC     0.722347
SLB   XOM     0.719726
dtype: float64


In [13]:
# CORRELATION STRATEGY: estimate hedge ratios on 30-min RETURNS.
#
# Key fix vs the cointegration attempt: the hedge ratio is now estimated
# at the SAME frequency we trade at. Regressing 30-min returns of stock i
# on stock j gives the beta that neutralizes their common short-term movement:
#     r_i = alpha + beta * r_j + residual
# The residual is the idiosyncratic divergence we trade.
#
# Estimated on TRAINING returns only — frozen for the test period.

alphas_cor={}
betas_cor={}
for (i, j) in top_N.index:
    returns_30min_train_i=returns_30min_train[i]
    returns_30min_train_j=returns_30min_train[j]
    model=LinearRegression()
    model.fit(returns_30min_train_j.values.reshape(-1,1),returns_30min_train_i)
    betas_cor[(i,j)]=model.coef_[0]
    alphas_cor[(i,j)]=model.intercept_

In [14]:
# CORRELATION STRATEGY: stateful signal generation on 30-min return residuals.
#
# Residual (signal source): the part of stock i's return not explained by stock j:
#     residual = r_i - (alpha + beta * r_j)
# When the residual z-score is stretched, i has diverged from j -> bet on convergence.
#
# Stateful position logic (positions persist, not bar-by-bar pulses):
#   FLAT:  enter short residual when z > +2, enter long when z < -2
#   SHORT: exit when z reverts below 0 (profit target) or blows past +3 (stop loss)
#   LONG:  exit when z reverts above 0 (profit target) or blows past -3 (stop loss)
#   NaN z-scores (rolling window warmup) -> stay flat
#
# P&L (tradable quantity): the beta-hedged pair return r_i - beta * r_j.
# Alpha is used in the signal residual but NOT in P&L — you can't earn an intercept.
# shift(1): position decided at t-1, earned on pair return at t — no look-ahead bias.
#
# Frozen alpha/beta from training; rolling z-score is backward-looking,
# so computing it fresh on the test set introduces no leakage.

cor_returns_dict_train = {}
cor_returns_dict_test = {}
window = 78  # 78 thirty-min bars = about 6 trading days

for (i, j), beta in betas_cor.items():
    alpha = alphas_cor[(i, j)]

    # ======================
    # Train
    # ======================
    # residual used for signal
    residual_train = returns_30min_train[i] - (alpha + beta * returns_30min_train[j])
    rolling_mean_train = residual_train.rolling(window=window).mean()
    rolling_std_train = residual_train.rolling(window=window).std()
    z_score_train = (residual_train - rolling_mean_train) / rolling_std_train

    positions_train = []
    position_train = 0
    for z in z_score_train:
        if pd.isna(z):
            positions_train.append(0)
            continue
        if position_train == 0:
            if z > 2:
                position_train = -1   # short residual
            elif z < -2:
                position_train = 1    # long residual
        elif position_train == -1:
            if z < 0 or z > 3:
                position_train = 0    # profit target or stop loss
        elif position_train == 1:
            if z > 0 or z < -3:
                position_train = 0    # profit target or stop loss
        positions_train.append(position_train)
    signal_train = pd.Series(positions_train, index=z_score_train.index)

    # tradable pair return (beta-hedged)
    pair_ret_train = returns_30min_train[i] - beta * returns_30min_train[j]
    # PnL: position decided at t-1, earned on pair return at t
    cor_returns_dict_train[f"{i}_{j}"] = signal_train.shift(1) * pair_ret_train

    # ======================
    # Test (frozen alpha/beta)
    # ======================
    residual_test = returns_30min_test[i] - (alpha + beta * returns_30min_test[j])
    rolling_mean_test = residual_test.rolling(window=window).mean()
    rolling_std_test = residual_test.rolling(window=window).std()
    z_score_test = (residual_test - rolling_mean_test) / rolling_std_test

    positions_test = []
    position_test = 0
    for z in z_score_test:
        if pd.isna(z):
            positions_test.append(0)
            continue
        if position_test == 0:
            if z > 2:
                position_test = -1
            elif z < -2:
                position_test = 1
        elif position_test == -1:
            if z < 0 or z > 3:
                position_test = 0
        elif position_test == 1:
            if z > 0 or z < -3:
                position_test = 0
        positions_test.append(position_test)
    signal_test = pd.Series(positions_test, index=z_score_test.index)

    pair_ret_test = returns_30min_test[i] - beta * returns_30min_test[j]
    cor_returns_dict_test[f"{i}_{j}"] = signal_test.shift(1) * pair_ret_test

cor_returns_train_df = pd.DataFrame(cor_returns_dict_train).dropna()
cor_returns_test_df = pd.DataFrame(cor_returns_dict_test).dropna()

In [15]:
# Portfolio construction with skfolio's RiskBudgeting optimizer (risk parity).
# Each pair is an asset; capital is allocated so every pair contributes
# equally to total portfolio variance — no concentration in any single pair.
# QuantStats tearsheet on the OOS portfolio returns.
# periods_per_year = 252 * 13 because we have 13 thirty-min bars per trading day —
# without this, Sharpe and volatility would be annualized as if bars were daily.
# Result: OOS annualized Sharpe ~0.71 vs ~0.2 for the cointegration approach —
# a ~3x improvement from matching the statistical method to the trading frequency.

model = RiskBudgeting(risk_measure=RiskMeasure.VARIANCE)
model.fit(cor_returns_train_df)
ptf = model.predict(cor_returns_test_df)
cor_returns_series = pd.Series(
    ptf.returns,
    index=cor_returns_test_df.index
)
qs.reports.html(
    cor_returns_series,
    output="pairs_trading_report_cor.html",
    periods_per_year=252 * 13
)

In [16]:
#Better suited for pairs trading where we have natural groupings(bank pairs, tech pairs, energy pairs)
#HierarchicalRiskParity knows that BAC/JPM and GS/JPM are related — they're both bank pairs. 
#It won't over-allocate to banks just because you have 3 bank pairs and 1 energy pair.
model = HierarchicalRiskParity()
model.fit(cor_returns_train_df)
ptf = model.predict(cor_returns_test_df)
cor_returns_series_hierarchical = pd.Series(ptf.returns, index=cor_returns_test_df.index)
qs.reports.html(cor_returns_series_hierarchical , output='pairs_trading_report_cor_hir.html',periods_per_year=252 * 13)

In [17]:
# Ledoit-Wolf shrinkage left OOS Sharpe essentially unchanged (0.71 vs 0.75) —
# expected: with ~117K training observations for a 20x20 covariance matrix,
# the sample estimate is already stable and there is little noise to shrink.

model_lw = RiskBudgeting(
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=EmpiricalPrior(covariance_estimator=LedoitWolf())
)
model_lw.fit(cor_returns_train_df)      # estimates SHRUNK covariance on train, solves weights
ptf_lw = model_lw.predict(cor_returns_test_df)   # applies frozen weights OOS

cor_returns_series_lw = pd.Series(
    ptf_lw.returns,
    index=cor_returns_test_df.index
)
qs.reports.html(
    cor_returns_series_lw,
    output="pairs_trading_report_cor_lw.html",
    periods_per_year=252 * 13
)